# Quattro modi di dire a una parola che cosa non deve guardare

*Anatomia di un LLM* · capitolo 5, L'attenzione e il blocco · script `03-04`, `scripts/03_04_maschere.py`

La maschera del capitolo precedente impedisce di vedere il futuro. Ce ne sono
altre tre di uso comune, che risolvono problemi diversi con lo stesso gesto:
mettere meno infinito in una casella prima della softmax.

Si esegue con `python3 scripts/03_04_maschere.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/03_04_maschere.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [1]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [2]:
import torch

from llm_anatomy.utils import prepara, titolo, verifica

prepara()

T = 10
punteggi = torch.randn(T, T)


def mostra(nome: str, m: torch.Tensor) -> None:
    print(f"  {nome}")
    for i in range(m.shape[0]):
        print("    " + " ".join("#" if v else "." for v in m[i].tolist()))


def pesi_da(m: torch.Tensor) -> torch.Tensor:
    return punteggi.masked_fill(~m, float("-inf")).softmax(dim=-1)

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


In [3]:
titolo("Non guardare avanti")
causale = torch.ones(T, T, dtype=torch.bool).tril()
mostra("riga = query, colonna = chiave", causale)
p = pesi_da(causale)
verifica(torch.triu(p, diagonal=1).abs().max().item() == 0.0,
         "nessun peso sopra la diagonale sopravvive al softmax")
print(f"  posizioni visibili: {int(causale.sum())} su {T * T} "
      f"({100 * float(causale.float().mean()):.0f}%)")


Non guardare avanti
-------------------
  riga = query, colonna = chiave
    # . . . . . . . . .
    # # . . . . . . . .
    # # # . . . . . . .
    # # # # . . . . . .
    # # # # # . . . . .
    # # # # # # . . . .
    # # # # # # # . . .
    # # # # # # # # . .
    # # # # # # # # # .
    # # # # # # # # # #
  [verifica superata] nessun peso sopra la diagonale sopravvive al softmax
  posizioni visibili: 55 su 100 (55%)


In [4]:
titolo("Non guardare il riempimento")
lunghezze = torch.tensor([10, 6, 3])
print(f"  tre sequenze nel batch, lunghezze {lunghezze.tolist()}")
valide = torch.arange(T)[None, :] < lunghezze[:, None]
padding = valide[:, None, :] & causale[None, :, :]
mostra("seconda sequenza (lunghezza 6)", padding[1])
print("  le colonne oltre la lunghezza reale sono spente su tutte le righe:")
print("  senza, il modello attenderebbe a token di riempimento")
p = pesi_da(padding[1])
verifica(p[:, 6:].abs().max().item() == 0.0,
         "nessun peso cade sulle posizioni di riempimento")


Non guardare il riempimento
---------------------------
  tre sequenze nel batch, lunghezze [10, 6, 3]
  seconda sequenza (lunghezza 6)
    # . . . . . . . . .
    # # . . . . . . . .
    # # # . . . . . . .
    # # # # . . . . . .
    # # # # # . . . . .
    # # # # # # . . . .
    # # # # # # . . . .
    # # # # # # . . . .
    # # # # # # . . . .
    # # # # # # . . . .
  le colonne oltre la lunghezza reale sono spente su tutte le righe:
  senza, il modello attenderebbe a token di riempimento
  [verifica superata] nessun peso cade sulle posizioni di riempimento


In [5]:
titolo("Guardare solo vicino")
finestra = 4
posizioni = torch.arange(T)
distanza = posizioni[:, None] - posizioni[None, :]   # riga meno colonna
scorrevole = causale & (distanza < finestra)
mostra(f"finestra di {finestra} posizioni", scorrevole)
print(f"  visibili: {int(scorrevole.sum())} contro {int(causale.sum())} della")
print("  causale piena: la memoria della KV cache smette di crescere con T")


Guardare solo vicino
--------------------
  finestra di 4 posizioni
    # . . . . . . . . .
    # # . . . . . . . .
    # # # . . . . . . .
    # # # # . . . . . .
    . # # # # . . . . .
    . . # # # # . . . .
    . . . # # # # . . .
    . . . . # # # # . .
    . . . . . # # # # .
    . . . . . . # # # #
  visibili: 34 contro 55 della
  causale piena: la memoria della KV cache smette di crescere con T


In [6]:
titolo("Non guardare l'altro documento")
documento = torch.tensor([0, 0, 0, 0, 1, 1, 1, 2, 2, 2])
per_documento = causale & (documento[:, None] == documento[None, :])
mostra("tre documenti concatenati", per_documento)
print("  senza questa maschera un documento leggerebbe la coda del")
print("  precedente, imparando correlazioni che non esistono")
p = pesi_da(per_documento)
verifica(p[7, :7].abs().max().item() == 0.0,
         "il primo token del terzo documento non vede i due precedenti")


Non guardare l'altro documento
------------------------------
  tre documenti concatenati
    # . . . . . . . . .
    # # . . . . . . . .
    # # # . . . . . . .
    # # # # . . . . . .
    . . . . # . . . . .
    . . . . # # . . . .
    . . . . # # # . . .
    . . . . . . . # . .
    . . . . . . . # # .
    . . . . . . . # # #
  senza questa maschera un documento leggerebbe la coda del
  precedente, imparando correlazioni che non esistono
  [verifica superata] il primo token del terzo documento non vede i due
                      precedenti


In [7]:
titolo("Si sommano")
composta = padding[1] & scorrevole & per_documento
mostra("riempimento + finestra + documento", composta)
print("  le maschere si compongono con un AND logico. Le righe oltre la")
print("  lunghezza reale della sequenza restano completamente spente: sono")
print("  posizioni di riempimento, e la loro uscita va scartata, non usata.")
righe_reali = composta[:int(lunghezze[1])]
verifica(bool(righe_reali.any(dim=1).all()),
         "ogni riga che corrisponde a un token vero conserva almeno una "
         "posizione")
cieche = int((~composta.any(dim=1)).sum())
print(f"  righe interamente spente: {cieche}, tutte oltre la posizione "
      f"{int(lunghezze[1])}")
print("  su una riga cieca il softmax di soli -infinito produce NaN: e' il")
print("  modo piu' comune di rompere un addestramento con le maschere, e si")
print("  evita escludendo quelle posizioni dalla perdita")


Si sommano
----------
  riempimento + finestra + documento
    # . . . . . . . . .
    # # . . . . . . . .
    # # # . . . . . . .
    # # # # . . . . . .
    . . . . # . . . . .
    . . . . # # . . . .
    . . . . # # . . . .
    . . . . . . . . . .
    . . . . . . . . . .
    . . . . . . . . . .
  le maschere si compongono con un AND logico. Le righe oltre la
  lunghezza reale della sequenza restano completamente spente: sono
  posizioni di riempimento, e la loro uscita va scartata, non usata.
  [verifica superata] ogni riga che corrisponde a un token vero conserva
                      almeno una posizione
  righe interamente spente: 3, tutte oltre la posizione 6
  su una riga cieca il softmax di soli -infinito produce NaN: e' il
  modo piu' comune di rompere un addestramento con le maschere, e si
  evita escludendo quelle posizioni dalla perdita


## Da provare

componi finestra scorrevole e causale, e stampa la maschera risultante.